<a href="https://colab.research.google.com/github/Bootcamp-IA-P6/P9E4/blob/bert_v3/notebooks/V3_bert/01_bert_toxicity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Comprobamos que la GPU está disponible
import torch

print(f"GPU disponible: {torch.cuda.is_available()}")
print(f"Nombre de la GPU: {torch.cuda.get_device_name(0)}")
print(f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Notebook BERT — Detección de Toxicidad con Transformers

Este notebook implementa un modelo basado en BERT
para detectar comentarios tóxicos en YouTube.

Es la versión avanzada del proyecto. Comparamos:
- Modelo clásico: TF-IDF + Regresión Logística
  → F1: 90.37% | AUC: 0.9724

- Modelo avanzado: BERT fine-tuned
  → Objetivo: superar esas métricas

## ¿Por qué BERT?
TF-IDF convierte palabras en números ignorando el contexto.
BERT entiende el significado completo de cada frase.

Ejemplo:
- "Black people are wonderful" → NO tóxico
- "Black people should die"    → TÓXICO

TF-IDF ve "black" y "people" en ambas frases.
BERT entiende que el contexto es completamente distinto.

## Hardware
- GPU: Tesla T4 (15.6 GB)
- Entorno: Google Colab

In [ ]:
# Instalamos la librería de HuggingFace para transformers
# En Colab se instala con pip, no con uv
!pip install transformers -q

# Manejo de datos
import pandas as pd
import numpy as np

# PyTorch: la librería de deep learning
import torch
from torch.utils.data import Dataset, DataLoader

# HuggingFace Transformers
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# Métricas
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    precision_score,
    classification_report
)

# Visualización
import matplotlib.pyplot as plt

# Verificamos GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo: {device}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

# 2. Carga del dataset

Cargamos el dataset enriquecido directamente desde
el repositorio de GitHub.

El dataset contiene 10.414 comentarios de tres fuentes:
- CSV original etiquetado por humanos
- Comentarios de YouTube etiquetados con Detoxify
- Comentarios tóxicos de Kaggle Jigsaw

In [ ]:
# URL raw del archivo en GitHub
# Raw significa el archivo en texto plano, no la página web de GitHub
URL_DATASET = (
    "https://raw.githubusercontent.com/"
    "Bootcamp-IA-P6/P9E4/eda_v3/"
    "data/raw/dataset_enriquecido.csv"
)

# Cargamos el dataset directamente desde GitHub
df = pd.read_csv(URL_DATASET)

# Verificamos que cargó correctamente
print(f"Filas:      {len(df)}")
print(f"Columnas:   {list(df.columns)}")
print(f"Tóxicos:    {df['IsToxic'].sum()} ({df['IsToxic'].mean()*100:.1f}%)")
print(f"No tóxicos: {(~df['IsToxic']).sum()} ({(~df['IsToxic']).mean()*100:.1f}%)")
print(f"\nPrimeras 3 filas:")
df.head(3)

# 3. Preparación de datos para BERT

BERT no necesita el pipeline de limpieza clásico.
No eliminamos stopwords ni lematizamos.
BERT entiende el contexto completo del texto original.

Lo que sí necesitamos:
1. Dividir en train y test
2. Tokenizar con el tokenizador de BERT
3. Crear un Dataset de PyTorch
4. Crear DataLoaders para el entrenamiento

In [ ]:
from sklearn.model_selection import train_test_split

# Convertimos IsToxic a números: True→1, False→0
# BERT necesita etiquetas numéricas, no booleanas
df['label'] = df['IsToxic'].astype(int)

# Dividimos en train y test
X_train, X_test, y_train, y_test = train_test_split(
    df['Text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"Train: {len(X_train)} comentarios")
print(f"Test:  {len(X_test)} comentarios")
print(f"Tóxicos en train: {sum(y_train)} ({sum(y_train)/len(y_train)*100:.1f}%)")
print(f"Tóxicos en test:  {sum(y_test)} ({sum(y_test)/len(y_test)*100:.1f}%)")

# Cargamos el tokenizador de BERT
# distilbert es una versión más ligera y rápida de BERT
# Mantiene el 97% del rendimiento con la mitad de parámetros
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Modelo: {MODEL_NAME}")
print(f"Vocabulario: {tokenizer.vocab_size} tokens")
print(f"Longitud máxima: {tokenizer.model_max_length}")

# Probamos el tokenizador con un ejemplo
ejemplo = "These people are disgusting and should be banned"
tokens  = tokenizer(ejemplo, return_tensors="pt")

print(f"Texto original: {ejemplo}")
print(f"Input IDs:      {tokens['input_ids']}")
print(f"Tokens:         {tokenizer.convert_ids_to_tokens(tokens['input_ids'][0])}")
print(f"Número de tokens: {tokens['input_ids'].shape[1]}")

# 4. Dataset de PyTorch

Creamos una clase Dataset personalizada que:
1. Tokeniza cada comentario con DistilBERT
2. Lo convierte en tensores de PyTorch
3. Lo prepara para moverlo a la GPU

In [ ]:
class ToxicDataset(Dataset):
    """
    Dataset personalizado para clasificación de toxicidad con BERT.
    Hereda de torch.utils.data.Dataset.
    """

    def __init__(self, textos, etiquetas, tokenizer, max_length=128):
        """
        Inicializa el dataset.

        Parámetros:
            textos:     lista de comentarios en texto
            etiquetas:  lista de etiquetas (0 o 1)
            tokenizer:  tokenizador de DistilBERT
            max_length: longitud máxima de cada comentario en tokens
        """
        self.textos    = textos
        self.etiquetas = etiquetas
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        """Devuelve el número total de ejemplos."""
        return len(self.textos)

    def __getitem__(self, idx):
        """
        Devuelve un ejemplo tokenizado listo para BERT.
        PyTorch llama a esta función automáticamente
        durante el entrenamiento.
        """
        texto    = str(self.textos[idx])
        etiqueta = self.etiquetas[idx]

        # Tokenizamos el texto con DistilBERT
        encoding = self.tokenizer(
            texto,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(etiqueta, dtype=torch.long)
        }

In [ ]:
# Creamos los datasets de train y test
train_dataset = ToxicDataset(X_train, y_train, tokenizer)
test_dataset  = ToxicDataset(X_test,  y_test,  tokenizer)

# Comprobamos que funcionan correctamente
ejemplo = train_dataset[0]
print("Claves del ejemplo:", list(ejemplo.keys()))
print("input_ids shape:   ", ejemplo['input_ids'].shape)
print("attention_mask shape:", ejemplo['attention_mask'].shape)
print("label:             ", ejemplo['labels'])
print(f"\nTotal train: {len(train_dataset)} ejemplos")
print(f"Total test:  {len(test_dataset)} ejemplos")

# 5. Modelo y configuración del entrenamiento

Cargamos DistilBERT preentrenado y lo adaptamos
para clasificación binaria (tóxico / no tóxico).

Fine-tuning: el modelo ya sabe inglés gracias al
preentrenamiento. Solo necesita aprender a distinguir
entre comentarios tóxicos y no tóxicos con nuestros datos.

In [ ]:
# Cargamos DistilBERT para clasificación de secuencias
# num_labels=2 → clasificación binaria: tóxico o no tóxico
modelo = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2
)

# Movemos el modelo a la GPU
modelo = modelo.to(device)

# Contamos los parámetros del modelo
total_params     = sum(p.numel() for p in modelo.parameters())
trainable_params = sum(p.numel() for p in modelo.parameters() if p.requires_grad)

print(f"Modelo: {MODEL_NAME}")
print(f"Parámetros totales:     {total_params:,}")
print(f"Parámetros entrenables: {trainable_params:,}")
print(f"Dispositivo: {next(modelo.parameters()).device}")

In [ ]:
def calcular_metricas(eval_pred):
    """
    Función que calcula las métricas durante la evaluación.
    El Trainer de HuggingFace la llama automáticamente.

    Parámetros:
        eval_pred: tupla (predicciones, etiquetas_reales)

    Devuelve:
        Diccionario con accuracy, f1, recall y precision
    """
    predicciones, etiquetas = eval_pred

    # Las predicciones son probabilidades → cogemos la clase con más probabilidad
    predicciones = predicciones.argmax(axis=1)

    return {
        'accuracy':  accuracy_score(etiquetas, predicciones),
        'f1':        f1_score(etiquetas, predicciones),
        'recall':    recall_score(etiquetas, predicciones),
        'precision': precision_score(etiquetas, predicciones)
    }

In [ ]:
# Configuración del entrenamiento
# Versión corregida para transformers >= 4.46
training_args = TrainingArguments(
    output_dir='./resultados_bert',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy='epoch',        # ← cambiado de evaluation_strategy
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_steps=50,
    warmup_steps=100,
    report_to='none'
)

print("Configuración del entrenamiento:")
print(f"  Épocas:           {training_args.num_train_epochs}")
print(f"  Batch size train: {training_args.per_device_train_batch_size}")
print(f"  Learning rate:    {training_args.learning_rate}")
print(f"  Weight decay:     {training_args.weight_decay}")

In [ ]:
trainer = Trainer(
    model=modelo,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=calcular_metricas
)

print("✅ Trainer configurado correctamente")
print(f"   Pasos por época: {len(train_dataset) // training_args.per_device_train_batch_size}")
print(f"   Pasos totales:   {len(train_dataset) // training_args.per_device_train_batch_size * training_args.num_train_epochs}")

# 6. Entrenamiento del modelo BERT

Lanzamos el fine-tuning de DistilBERT con
nuestros 10.414 comentarios.

El Trainer gestiona automáticamente:
- El bucle de entrenamiento
- El cálculo de gradientes
- La evaluación al final de cada época
- El guardado del mejor modelo

In [ ]:
# Lanzamos el entrenamiento
print("Iniciando entrenamiento...")
print(f"Épocas: {training_args.num_train_epochs}")
print(f"Pasos totales: 780")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print("=" * 50)

resultado_entrenamiento = trainer.train()

print("\n" + "=" * 50)
print("✅ Entrenamiento completado")
print(f"Tiempo total: {resultado_entrenamiento.metrics['train_runtime']:.0f} segundos")
print(f"Pasos por segundo: {resultado_entrenamiento.metrics['train_steps_per_second']:.2f}")

# 7. Evaluación detallada del modelo BERT

Evaluamos el modelo final (época 2) con el conjunto
de test completo y generamos la matriz de confusión
y el classification report para comparar con el
modelo clásico.

In [ ]:
# Obtenemos las predicciones del modelo final
print("Generando predicciones...")
predicciones_output = trainer.predict(test_dataset)

# Extraemos las probabilidades y convertimos a clases
probabilidades = predicciones_output.predictions
y_pred_bert    = probabilidades.argmax(axis=1)
y_real         = predicciones_output.label_ids

print(f"Total predicciones: {len(y_pred_bert)}")
print(f"Tóxicos predichos:  {y_pred_bert.sum()}")
print(f"Tóxicos reales:     {y_real.sum()}")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# Calculamos todas las métricas
acc_bert  = accuracy_score(y_real, y_pred_bert)
f1_bert   = f1_score(y_real, y_pred_bert)
rec_bert  = recall_score(y_real, y_pred_bert)
prec_bert = precision_score(y_real, y_pred_bert)

print("=" * 55)
print("   RESULTADOS FINALES — DistilBERT")
print("=" * 55)
print(f"Accuracy:  {acc_bert*100:.2f}%")
print(f"F1 Score:  {f1_bert*100:.2f}%")
print(f"Recall:    {rec_bert*100:.2f}%")
print(f"Precision: {prec_bert*100:.2f}%")
print("=" * 55)

# Classification report completo
print("\nClassification Report:")
print(classification_report(
    y_real,
    y_pred_bert,
    target_names=['No tóxico', 'Tóxico']
))

In [ ]:
# Matriz de confusión
cm = confusion_matrix(y_real, y_pred_bert)

fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['No tóxico', 'Tóxico'],
    yticklabels=['No tóxico', 'Tóxico'],
    ax=ax
)

ax.set_title('Matriz de confusión — DistilBERT')
ax.set_xlabel('Predicción del modelo')
ax.set_ylabel('Valor real')

plt.tight_layout()
plt.show()

# Extraemos los valores
vn, fp, fn, vp = cm.ravel()
print(f"\nVN (no tóxico correcto):  {vn}")
print(f"FP (falsa alarma):         {fp}")
print(f"FN (tóxico no detectado):  {fn}")
print(f"VP (tóxico detectado):     {vp}")
print(f"\nDe {vp+fn} tóxicos reales, detectamos {vp} ({vp/(vp+fn)*100:.1f}%)")
print(f"De {vp+fn} tóxicos reales, se escaparon {fn} ({fn/(vp+fn)*100:.1f}%)")

In [ ]:
# ── Comparativa completa TF-IDF vs BERT ──────────────
print("=" * 65)
print("   COMPARATIVA FINAL: TF-IDF vs DistilBERT")
print("=" * 65)
print(f"{'Métrica':<14} {'TF-IDF + LogReg':>16} {'DistilBERT':>12} {'Mejora':>10}")
print("-" * 65)

metricas = [
    ("Accuracy",  90.88, acc_bert*100),
    ("F1 Score",  90.37, f1_bert*100),
    ("Recall",    87.27, rec_bert*100),
    ("Precision", 93.69, prec_bert*100),
]

for nombre, clasico, bert in metricas:
    mejora = bert - clasico
    icono  = "✅" if mejora > 0 else "⚠️"
    print(
        f"{nombre:<14} "
        f"{clasico:>15.2f}% "
        f"{bert:>11.2f}% "
        f"{mejora:>+9.2f}p {icono}"
    )

print("=" * 65)
print(f"\n{'Modelo clásico':<20} → simple, rápido, sin GPU")
print(f"{'DistilBERT':<20} → más preciso, necesita GPU, 5 minutos")

In [ ]:
# Calculamos el overfitting de BERT
# Necesitamos las métricas en train

predicciones_train  = trainer.predict(train_dataset)
y_pred_train_bert   = predicciones_train.predictions.argmax(axis=1)
y_real_train        = predicciones_train.label_ids

acc_train_bert = accuracy_score(y_real_train, y_pred_train_bert)
f1_train_bert  = f1_score(y_real_train, y_pred_train_bert)

diff_acc = abs(acc_train_bert - acc_bert) * 100
diff_f1  = abs(f1_train_bert  - f1_bert)  * 100

print("=" * 55)
print("   CONTROL DE OVERFITTING — DistilBERT")
print("=" * 55)
print(f"{'Métrica':<12} {'Train':>8} {'Test':>8} {'Diferencia':>12}")
print("-" * 55)
print(
    f"{'Accuracy':<12} "
    f"{acc_train_bert*100:>7.2f}% "
    f"{acc_bert*100:>7.2f}% "
    f"{diff_acc:>10.2f}p"
)
print(
    f"{'F1 Score':<12} "
    f"{f1_train_bert*100:>7.2f}% "
    f"{f1_bert*100:>7.2f}% "
    f"{diff_f1:>10.2f}p"
)
print("-" * 55)

limite = 5.0
if diff_acc < limite and diff_f1 < limite:
    print("✅ Sin overfitting significativo")
else:
    print(f"⚠️  Diferencia supera {limite} puntos")
print("=" * 55)

# Comparativa de overfitting con el modelo clásico
print(f"\nComparativa de overfitting:")
print(f"  TF-IDF + LogReg: 3.92p ✅")
print(f"  DistilBERT:      {diff_acc:.2f}p {'✅' if diff_acc < 5 else '⚠️'}")

In [ ]:
# Probamos con comentarios reales
def predecir_bert(texto, modelo, tokenizer, device):
    """
    Predice si un comentario es tóxico usando DistilBERT.

    Parámetros:
        texto:     el comentario a analizar
        modelo:    modelo DistilBERT fine-tuned
        tokenizer: tokenizador de DistilBERT
        device:    cuda o cpu

    Devuelve:
        etiqueta y confianza
    """
    # Tokenizamos el texto
    encoding = tokenizer(
        texto,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    ).to(device)

    # Predecimos sin calcular gradientes
    # torch.no_grad() ahorra memoria GPU durante la inferencia
    with torch.no_grad():
        output = modelo(**encoding)

    # Convertimos a probabilidades con softmax
    probabilidades = torch.softmax(output.logits, dim=1)
    prediccion     = probabilidades.argmax().item()
    confianza      = probabilidades.max().item() * 100

    resultado = "🔴 TÓXICO" if prediccion == 1 else "🟢 NO TÓXICO"
    return resultado, confianza


# Ejemplos de prueba
ejemplos = [
    "I love how diverse and respectful this community is",
    "These people are disgusting and should be banned forever",
    "Black people are wonderful and deserve equal rights",
    "Black people should be eliminated from this country",
    "The police did a great job protecting everyone",
    "All cops are racist murderers who deserve to die",
]

print("Prueba con ejemplos reales:")
print("=" * 65)

for comentario in ejemplos:
    resultado, confianza = predecir_bert(
        comentario, modelo, tokenizer, device
    )
    print(f"Texto:     {comentario[:55]}...")
    print(f"Resultado: {resultado} (confianza: {confianza:.1f}%)")
    print()

# 8. Guardado del modelo y conclusiones

Guardamos el modelo fine-tuned en Google Drive
para poder reutilizarlo sin reentrenar.

El entrenamiento tardó 5 minutos con GPU T4.
Sin GPU tardaría entre 3 y 5 horas.

In [ ]:
from google.colab import drive

# Montamos Google Drive en Colab
drive.mount('/content/drive')

print("✅ Google Drive montado correctamente")

In [ ]:
import os

# Creamos la carpeta en Google Drive
ruta_modelo = '/content/drive/MyDrive/P9E4_BERT/modelo_final'
os.makedirs(ruta_modelo, exist_ok=True)

# Guardamos el modelo y el tokenizador
modelo.save_pretrained(ruta_modelo)
tokenizer.save_pretrained(ruta_modelo)

print(f"✅ Modelo guardado en: {ruta_modelo}")
print(f"\nArchivos guardados:")
for archivo in os.listdir(ruta_modelo):
    tamanyo = os.path.getsize(
        os.path.join(ruta_modelo, archivo)
    ) / (1024 * 1024)
    print(f"  → {archivo} ({tamanyo:.1f} MB)")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Cargamos el modelo guardado para verificar
print("Verificando que el modelo guardado funciona...")

modelo_cargado     = AutoModelForSequenceClassification.from_pretrained(ruta_modelo)
tokenizer_cargado  = AutoTokenizer.from_pretrained(ruta_modelo)
modelo_cargado     = modelo_cargado.to(device)

# Probamos una predicción
texto_prueba = "These people are absolutely disgusting"
resultado, confianza = predecir_bert(
    texto_prueba,
    modelo_cargado,
    tokenizer_cargado,
    device
)

print(f"✅ Modelo cargado correctamente")
print(f"\nPrueba de verificación:")
print(f"Texto:     {texto_prueba}")
print(f"Resultado: {resultado} (confianza: {confianza:.1f}%)")

# 9. Conclusiones del notebook BERT

## ¿Qué hicimos?

Implementamos fine-tuning de DistilBERT para detectar
comentarios tóxicos en YouTube comparándolo con el
pipeline clásico TF-IDF + Regresión Logística.

---

## Resultados obtenidos

| Métrica   | TF-IDF + LogReg | DistilBERT | Mejora  |
|-----------|-----------------|------------|---------|
| Accuracy  | 90.88%          | 95.63%     | +4.75p  |
| F1 Score  | 90.37%          | 95.53%     | +5.16p  |
| Recall    | 87.27%          | 95.20%     | +7.93p  |
| Precision | 93.69%          | 95.86%     | +2.17p  |
| Overfitting| 3.92p          | 2.76p      | -1.16p  |

---

## ¿Por qué BERT es mejor?

TF-IDF convierte palabras en números ignorando el contexto.
Cada palabra es independiente de las demás.

BERT entiende el significado completo de cada frase:
- "Black people are wonderful" → NO tóxico ✓
- "Black people should be eliminated" → casi tóxico ✓
- "All cops are racist murderers" → TÓXICO ✓

---

## Limitaciones encontradas

1. Necesita GPU para ser viable en tiempo razonable.
   Sin GPU: 3-5 horas. Con GPU T4: 5 minutos.

2. El lenguaje indirecto y eufemismos son difíciles:
   "eliminated from this country" → confianza baja (56.5%)
   En producción estos casos requerirían revisión manual.

3. Dependencia de Colab:
   No se puede ejecutar localmente sin GPU dedicada.

---

## Comparativa de recursos

| Aspecto          | TF-IDF + LogReg | DistilBERT     |
|------------------|-----------------|----------------|
| Tiempo entreno   | Segundos        | 5 minutos      |
| GPU necesaria    | No              | Sí             |
| Parámetros       | ~5.000          | 67 millones    |
| F1 Score         | 90.37%          | 95.53%         |
| Interpretabilidad| Alta            | Baja           |

---

## Conclusión principal

BERT supera al modelo clásico en todas las métricas
con una mejora de +5.16p en F1 y +7.93p en Recall.

El overfitting es menor (2.76p vs 3.92p) a pesar de
tener 67 millones de parámetros, gracias al preentrenamiento
con enormes cantidades de texto.

Para un sistema de moderación real de YouTube,
DistilBERT es la mejor opción si se dispone de GPU.
Para un sistema sin GPU, TF-IDF + LogReg es suficiente
y mucho más eficiente en recursos.

---

## Archivos generados

| Archivo | Ubicación | Contenido |
|---|---|---|
| modelo_final/ | Google Drive/P9E4_BERT/ | Modelo DistilBERT fine-tuned |
| 01_bert_toxicity.ipynb | GitHub/notebooks/V3_bert/ | Este notebook |